# 12 — Batch Pipeline

**Marker:** `NOTEBOOK_12_BATCH_PIPELINE_FRESH_V1`

This notebook ties the working project together.

It starts with the knowledge tree from Notebook 02, generates topic candidates,
lets you choose which candidates to run, and then orchestrates:

1. outline generation
2. script generation
3. script editing
4. fact checking
5. metadata
6. Kokoro narration
7. captions
8. final video assembly

The default first run processes **one selected topic** and reuses your single
Subway Surfers recording.


## Load the project

In [1]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "educational_shorts").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root containing educational_shorts/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from educational_shorts.batch import (
    BatchConfig,
    preflight_batch_pipeline,
    prepare_topic_batch,
    run_batch_pipeline,
    summarize_batch_manifest,
    summarize_topic_plan,
)

print("NOTEBOOK_12_BATCH_PIPELINE_FRESH_V1")
print(f"Project root: {PROJECT_ROOT}")

NOTEBOOK_12_BATCH_PIPELINE_FRESH_V1
Project root: c:\Users\hitch\python_files\educational_shorts


## Configuration

For the first test, keep `SELECTED_TOPIC_INDEXES = [0]`.

`PAUSE_ON_MANUAL_REVIEW = True` prevents a questionable script from
automatically becoming a public-facing video. It still saves the checked script
and fact-check report for inspection.


In [ ]:
ROOT_CATEGORY = "Science"
TREE_FILENAME = None

# Set to a full path such as ["Science", "Biology", "Microbiology"]
# to bypass random category selection. Leave as None for random selection.
CATEGORY_PATH_OVERRIDE = None

TOPIC_CANDIDATE_COUNT = 10
MIN_CATEGORY_DEPTH = 2
MAX_CATEGORY_DEPTH = 3

CATEGORY_SELECTION_SEED = 42
TOPIC_GENERATION_SEED = 42

# Choose candidate indexes after previewing the next section.
SELECTED_TOPIC_INDEXES = [0]

PAUSE_ON_MANUAL_REVIEW = False
CONTINUE_ON_ERROR = False
SKIP_EXISTING_FINAL = True

GAMEPLAY_SUBDIRECTORY = "subway_surfers"
BACKGROUND_VIDEO_FILENAME = None

TTS_VOICE = "am_michael"
TTS_SPEED = 1.0

CAPTION_STYLE = "phrase"
ASS_FONT_NAME = "Arial"
ASS_FONT_SIZE = 72
ASS_MARGIN_V = 300

OUTPUT_WIDTH = 1080
OUTPUT_HEIGHT = 1920
CROP_ANCHOR_Y = "top"
LOOP_BACKGROUND = True

config = BatchConfig(
    root_category=ROOT_CATEGORY,
    tree_filename=TREE_FILENAME,
    category_path_override=CATEGORY_PATH_OVERRIDE,
    topic_candidate_count=TOPIC_CANDIDATE_COUNT,
    min_category_depth=MIN_CATEGORY_DEPTH,
    max_category_depth=MAX_CATEGORY_DEPTH,
    category_selection_seed=CATEGORY_SELECTION_SEED,
    topic_generation_seed=TOPIC_GENERATION_SEED,
    pause_on_manual_review=PAUSE_ON_MANUAL_REVIEW,
    gameplay_subdirectory=GAMEPLAY_SUBDIRECTORY,
    background_video_filename=BACKGROUND_VIDEO_FILENAME,
    tts_voice=TTS_VOICE,
    tts_speed=TTS_SPEED,
    caption_style=CAPTION_STYLE,
    ass_font_name=ASS_FONT_NAME,
    ass_font_size=ASS_FONT_SIZE,
    ass_margin_v=ASS_MARGIN_V,
    output_width=OUTPUT_WIDTH,
    output_height=OUTPUT_HEIGHT,
    crop_anchor_y=CROP_ANCHOR_Y,
    loop_background=LOOP_BACKGROUND,
    skip_existing_final=SKIP_EXISTING_FINAL,
    continue_on_error=CONTINUE_ON_ERROR,
)

print(config.model_dump_json(indent=2))

{
  "root_category": "Science",
  "tree_filename": null,
  "category_path_override": null,
  "topic_candidate_count": 10,
  "min_category_depth": 2,
  "max_category_depth": 3,
  "category_selection_seed": 42,
  "topic_generation_seed": 42,
  "topic_generation_temperature": 0.7,
  "target_seconds": 60,
  "section_count": 4,
  "outline_temperature": 0.4,
  "outline_seed": 42,
  "target_words_per_minute": 145,
  "script_temperature": 0.5,
  "script_seed": 42,
  "editor_minimum_seconds": 40,
  "editor_maximum_seconds": 60,
  "editor_temperature": 0.4,
  "editor_seed": 42,
  "fact_check_minimum_words": 105,
  "fact_check_maximum_words": 150,
  "fact_check_temperature": 0.1,
  "fact_check_seed": 42,
  "metadata_temperature": 0.5,
  "metadata_seed": 42,
  "metadata_max_attempts": 4,
  "pause_on_manual_review": true,
  "tts_language_code": "a",
  "tts_voice": "am_michael",
  "tts_speed": 1.0,
  "tts_chunk_pause_ms": 80,
  "tts_segment_pause_ms": 240,
  "tts_target_peak_dbfs": -1.0,
  "normaliz

## Preflight check

In [3]:
preflight = preflight_batch_pipeline(
    project_root=PROJECT_ROOT,
    config=config,
)

for name, value in preflight.items():
    print(f"{name}: {value}")

project_root: c:\Users\hitch\python_files\educational_shorts
tree_path: c:\Users\hitch\python_files\educational_shorts\data\knowledge_tree\science.json
tree_root: Science
gameplay_directory: c:\Users\hitch\python_files\educational_shorts\data\gameplay\subway_surfers
background_video: c:\Users\hitch\python_files\educational_shorts\data\gameplay\subway_surfers\ScreenRecording_07-23-2026 13-07-17_1.mp4
ffmpeg: C:\Users\hitch\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1.2-full_build\bin\ffmpeg.EXE
ffprobe: C:\Users\hitch\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1.2-full_build\bin\ffprobe.EXE


## Generate topic candidates

This only generates and saves candidate topics. It does **not** yet create
scripts, audio, captions, or videos.


In [4]:
topic_plan = prepare_topic_batch(
    project_root=PROJECT_ROOT,
    config=config,
)

for name, value in summarize_topic_plan(topic_plan).items():
    print(f"{name}: {value}")

candidate_run_id: 20260723T183655127441Z
category_path: Science > Biology > Microbiology
candidate_count: 10
topics_file: c:\Users\hitch\python_files\educational_shorts\data\topics\science__biology__microbiology__20260723T183655127441Z.json


## Preview candidates

In [5]:
print("CATEGORY:")
print(" > ".join(topic_plan.category_path))
print()

for index, topic in enumerate(topic_plan.topics.topics):
    print(f"[{index}] {topic.title}")
    print(f"    {topic.learning_objective}")
    print()

CATEGORY:
Science > Biology > Microbiology

[0] How Bacteria Communicate
    Understand how bacteria use chemical signals to coordinate behavior.

[1] The Role of Biofilms in Disease
    Explore how biofilms contribute to antibiotic resistance and infection persistence.

[2] Comparing Viruses and Bacteria
    Differentiate between viruses and bacteria based on structure, reproduction, and treatment.

[3] How Antimicrobial Resistance Develops
    Explain the mechanisms by which microbes develop resistance to antibiotics.

[4] The Microbial World in Your Gut
    Discover how gut microbiota influence digestion and immune health.

[5] How Viruses Infect Cells
    Learn the step-by-step process of viral entry, replication, and cell damage.

[6] The Life Cycle of a Bacteriophage
    Understand how bacteriophages infect bacteria and replicate within them.

[7] Why Some Microbes Are Beneficial
    Identify the roles of beneficial microbes in ecosystems and human health.

[8] The Impact of UV L

## Confirm selected indexes

Edit `SELECTED_TOPIC_INDEXES` in the configuration cell, rerun that cell, and
then run this cell.

For the first complete test, one index is enough:

```python
SELECTED_TOPIC_INDEXES = [0]
```


In [6]:
print(f"Selected indexes: {SELECTED_TOPIC_INDEXES}")
print()

for index in SELECTED_TOPIC_INDEXES:
    topic = topic_plan.topics.topics[index]
    print(f"[{index}] {topic.title}")

Selected indexes: [0]

[0] How Bacteria Communicate


## Run the selected topics

This is the long-running cell. It may make multiple local Ollama calls, run
Kokoro, create captions, and invoke FFmpeg.

The Kokoro synthesizer is loaded once and reused when several topics are
selected.


In [ ]:
batch_manifest = run_batch_pipeline(
    project_root=PROJECT_ROOT,
    plan=topic_plan,
    selected_topic_indexes=SELECTED_TOPIC_INDEXES,
    config=config,
)

for name, value in summarize_batch_manifest(
    batch_manifest
).items():
    print(f"{name}: {value}")


ITEM 1/1: How Bacteria Communicate
Attempt 1: generated 92 words. Minimum is 105. Retrying...
Retrieving 1/6: Bacteria use chemicals to communicate.
  3 source(s) from web
Retrieving 2/6: Bacteria are social microbes that exchange signals through chemical messengers.
  3 source(s) from web
Retrieving 3/6: Quorum sensing is how bacteria 'talk' using molecules they release.
  3 source(s) from web
Retrieving 4/6: When enough signals are present, bacteria know they’re in a crowd and can act as a group.
  3 source(s) from web
Retrieving 5/6: Bacterial communication helps them survive by coordinating actions like building biofilms.
  3 source(s) from web
Retrieving 6/6: Studying bacterial communication helps scientists create new antibiotics and apply bacteria in environmental solutions.
  3 source(s) from web
Verifying claim 1/6: Bacteria use chemicals to communicate.
Verifying claim 2/6: Bacteria are social microbes that exchange signals through chemical messengers.
Verifying claim 3/6: Q

## Review each topic result

In [8]:
for item in batch_manifest.items:
    print("=" * 72)
    print(f"{item.item_number}. {item.topic_title}")
    print(f"Status: {item.status}")
    print(f"Final stage: {item.final_stage}")
    print(f"Message: {item.message}")
    print(
        "Fact-check verdict: "
        f"{item.fact_check_verdict}"
    )
    print(
        "Requires manual review: "
        f"{item.requires_manual_review}"
    )
    print("Completed stages:")
    for stage in item.stages_completed:
        print(f"  - {stage}")

    print("Paths:")
    for name, path in item.paths.items():
        print(f"  {name}: {path}")
    print()

1. How Bacteria Communicate
Status: manual_review
Final stage: fact_checking
Message: Fact checking completed, but the topic was held before publication assets because manual review is enabled.
Fact-check verdict: manual_review
Requires manual review: True
Completed stages:
  - outline_generation
  - script_generation
  - script_editing
  - fact_checking
Paths:
  outline: c:\Users\hitch\python_files\educational_shorts\data\outlines\how_bacteria_communicate.json
  script: c:\Users\hitch\python_files\educational_shorts\data\scripts\how_bacteria_communicate.json
  edited_script: c:\Users\hitch\python_files\educational_shorts\data\edited_scripts\how_bacteria_communicate.json
  checked_script: c:\Users\hitch\python_files\educational_shorts\data\checked_scripts\how_bacteria_communicate.json
  fact_check_report: c:\Users\hitch\python_files\educational_shorts\data\fact_checks\how_bacteria_communicate_fact_check.json



## Preview completed videos

In [9]:
from IPython.display import Video, display

completed_items = [
    item
    for item in batch_manifest.items
    if item.status in {"completed", "skipped_existing"}
    and item.paths.get("final_video")
]

if not completed_items:
    print(
        "No completed videos to preview. Check whether a topic was held "
        "for manual review or failed at an earlier stage."
    )
else:
    for item in completed_items:
        print(item.topic_title)
        display(
            Video(
                filename=item.paths["final_video"],
                embed=True,
                width=360,
            )
        )

No completed videos to preview. Check whether a topic was held for manual review or failed at an earlier stage.


## Inspect the saved batch manifest

In [10]:
batch_manifest_path = (
    Path(batch_manifest.output_directory)
    / batch_manifest.manifest_filename
)

print(f"Saved manifest: {batch_manifest_path}")
print()
print(
    batch_manifest_path.read_text(
        encoding="utf-8"
    )
)

Saved manifest: c:\Users\hitch\python_files\educational_shorts\data\batch_runs\20260723T183655241695Z\batch_manifest.json

{
  "marker": "NOTEBOOK_12_BATCH_PIPELINE_FRESH_V1",
  "run_id": "20260723T183655241695Z",
  "category_path": [
    "Science",
    "Biology",
    "Microbiology"
  ],
  "selected_topic_indexes": [
    0
  ],
  "selected_topic_titles": [
    "How Bacteria Communicate"
  ],
  "status": "completed_with_holds",
  "output_directory": "c:\\Users\\hitch\\python_files\\educational_shorts\\data\\batch_runs\\20260723T183655241695Z",
  "manifest_filename": "batch_manifest.json",
  "started_at_utc": "2026-07-23T18:36:55.241695+00:00",
  "finished_at_utc": "2026-07-23T19:17:28.508532+00:00",
  "config": {
    "root_category": "Science",
    "tree_filename": null,
    "category_path_override": null,
    "topic_candidate_count": 10,
    "min_category_depth": 2,
    "max_category_depth": 3,
    "category_selection_seed": 42,
    "topic_generation_seed": 42,
    "topic_generation_te